# Oviva Data Scientist Task
### Yagmur Dalman


In [ ]:
%%capture
!pip -q install pymc arviz

import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# point to the files:
activations_path = '/content/drive/MyDrive/activations.csv'
interventions_path = '/content/drive/MyDrive/interventions.csv'

acts_raw = pd.read_csv(activations_path)
intr_raw = pd.read_csv(interventions_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
acts_raw.head()

,school_id,date
0,0,2024-01-09
1,0,2024-01-15
2,0,2024-01-24
3,0,2024-01-31
4,0,2024-02-08


In [ ]:
intr_raw.head()

,school_id,date,intervention
0,0,2024-08-02,welcome_kit
1,0,2024-09-14,virtual_workshop
2,0,2024-11-14,onsite_school_visit
3,1,2024-06-17,teacher_webinar
4,2,2024-08-15,email_nudge


In [ ]:
# normalize column names

df_activations = acts_raw.rename(columns = {"school_id": "client_id"}).copy()
df_interventions = intr_raw.rename(columns = {"school_id": "client_id"}).copy()

df_activations["date"] = pd.to_datetime(df_activations["date"], errors="coerce")
df_interventions["date"] = pd.to_datetime(df_interventions["date"], errors="coerce")

In [ ]:
print(df_activations.shape, df_interventions.shape)
display(df_activations.head(5))
display(df_interventions.head(5))

(10553, 2) (847, 3)


,client_id,date
0,0,2024-01-09
1,0,2024-01-15
2,0,2024-01-24
3,0,2024-01-31
4,0,2024-02-08


,client_id,date,intervention
0,0,2024-08-02,welcome_kit
1,0,2024-09-14,virtual_workshop
2,0,2024-11-14,onsite_school_visit
3,1,2024-06-17,teacher_webinar
4,2,2024-08-15,email_nudge


In [ ]:
# Time window: Data spans 2024-01-01 to 2024-12-31, fall back to observed min/max if needed

START = pd.Timestamp("2024-01-01")
END = pd.Timestamp("2024-12-31")
START = min(START, pd.concat([df_activations.date, df_interventions.date], ignore_index = True).min())
END = max(END, pd.concat([df_activations.date, df_interventions.date], ignore_index = True).max())

In [ ]:
# sale events (one row per activation)
df_sales = df_activations[["client_id","date"]].copy()
df_sales["event"] = "sale"

# intervention events
df_interventions = df_interventions[["client_id","date","intervention"]].copy()
df_interventions["event"] = "intervention"

# put all events together
df_events = pd.concat([df_sales, df_interventions], ignore_index=True)
#df_events.head()

# add a dummy "end-of-window" event so we capture exposure after the last real event
end_rows = (
    df_events.groupby("client_id", as_index=False)["date"].max()
          .assign(date=lambda x: END + pd.Timedelta(days=1), event="end")
)
df_events = pd.concat([df_events, end_rows], ignore_index=True)

# sort by client/date
df_events = df_events.sort_values(["client_id","date","event"]).reset_index(drop=True)

df_events.head()

,client_id,date,event,intervention
0,0,2024-01-09,sale,NaN
1,0,2024-01-15,sale,NaN
2,0,2024-01-24,sale,NaN
3,0,2024-01-31,sale,NaN
4,0,2024-02-08,sale,NaN


In [ ]:
# for each intervention type, build an "active" flag that flips to 1 at (and after) the first occurrence
intervention_types = sorted(df_interventions["intervention"].dropna().unique().tolist())


for itype in intervention_types:
    # is_{itype}_event == 1 iff this row is an intervention AND its type equals `itype`
    # (True/False converted to 1/0)
    df_events[f"is_{itype}_event"] = (
        df_events["event"].eq("intervention") & df_events["intervention"].eq(itype)
    ).astype(int)

# carry the effect forward in time: once triggered, it stays "on" (no-decay)

def propagate_active_flags(df):
    """
    df: the timeline (sorted rows) for a single client_id.

    Goal:
      Turn the one-off "is_{itype}_event" into a persistent "active_{itype}" flag:
      once an intervention type happens at least once, it remains 1 for all *subsequent* rows.
    """
    for itype in intervention_types:
        # cumsum turns sequences like 0,0,1,0,0,1,... into 0,0,1,1,1,2,...
        # clip(upper=1) turns everything above 1 into exactly 1, i.e., a cumulative OR.
        active = df[f"is_{itype}_event"].cumsum().clip(upper=1).astype(int)
        df[f"active_{itype}"] = active

    return df

# apply per client_id so each school's state evolves independently
df_events = df_events.groupby("client_id", group_keys=False).apply(propagate_active_flags)


/tmp/ipython-input-3193779360.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_events = df_events.groupby("client_id", group_keys=False).apply(propagate_active_flags)


In [ ]:
# exposure and state per interval: compute dt and the state at interval start

def add_dt(df):
    """
    df: the timeline for a single client_id - already sorted

    Outputs added:
      - dt: days since the previous row (the exposure length for this interval)
      - is_sale: whether the current row is a sale event (1) or not (0)
      - state_{itype}: whether the intervention `itype` was ACTIVE during this interval
      We use the previous row's 'active' flag, because interval state is defined at interval start
    """
    # previous timestamp - the very first interval starts at START (the window start)
    prev = df["date"].shift(1).fillna(START)

    # exposure length in whole days
    df["dt"] = (df["date"] - prev).dt.days.astype(float)

    # mark sales (interventions and the 'end' marker are not sales)
    df["is_sale"] = (df["event"] == "sale").astype(int)

    # interval state = what was active at the *start* of this interval:
    # shift(1) pulls the previous row's active flags forward.
    for itype in intervention_types:
        df[f"state_{itype}"] = df[f"active_{itype}"].shift(1).fillna(0).astype(int)

    return df

df_events = df_events.groupby("client_id", group_keys=False).apply(add_dt)



/tmp/ipython-input-3148019425.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_events = df_events.groupby("client_id", group_keys=False).apply(add_dt)


In [ ]:

# clean up edge cases

# drop zero-length intervals (dt == 0) if any show up (e.g., multiple rows with identical timestamps)
df_events = df_events[df_events["dt"] > 0].copy()

# The final 'end' row contributes exposure (dt) but never a sale; we keep it as is


df_events.head(10)

,client_id,date,event,intervention,is_email_nudge_event,is_onsite_school_visit_event,is_teacher_webinar_event,is_virtual_workshop_event,is_welcome_kit_event,active_email_nudge,active_onsite_school_visit,active_teacher_webinar,active_virtual_workshop,active_welcome_kit,dt,is_sale,state_email_nudge,state_onsite_school_visit,state_teacher_webinar,state_virtual_workshop,state_welcome_kit
0,0,2024-01-09,sale,NaN,0,0,0,0,0,0,0,0,0,0,8.0,1,0,0,0,0,0
1,0,2024-01-15,sale,NaN,0,0,0,0,0,0,0,0,0,0,6.0,1,0,0,0,0,0
2,0,2024-01-24,sale,NaN,0,0,0,0,0,0,0,0,0,0,9.0,1,0,0,0,0,0
3,0,2024-01-31,sale,NaN,0,0,0,0,0,0,0,0,0,0,7.0,1,0,0,0,0,0
4,0,2024-02-08,sale,NaN,0,0,0,0,0,0,0,0,0,0,8.0,1,0,0,0,0,0
5,0,2024-02-23,sale,NaN,0,0,0,0,0,0,0,0,0,0,15.0,1,0,0,0,0,0
6,0,2024-03-07,sale,NaN,0,0,0,0,0,0,0,0,0,0,13.0,1,0,0,0,0,0
7,0,2024-03-10,sale,NaN,0,0,0,0,0,0,0,0,0,0,3.0,1,0,0,0,0,0
8,0,2024-03-16,sale,NaN,0,0,0,0,0,0,0,0,0,0,6.0,1,0,0,0,0,0
9,0,2024-04-20,sale,NaN,0,0,0,0,0,0,0,0,0,0,35.0,1,0,0,0,0,0


In [ ]:
#  aggregate to counts + exposure per school + intervention-state
""" compress many per-interval rows into *sufficient statistics* for a Poisson model:
      for each (school, state combo) we want:
        - n  = total number of sales observed under that state
        - dt = total exposure time (in days) under that state
"""

group_cols = ["client_id"] + [f"state_{itype}" for itype in intervention_types]

data = (
    df_events
    .groupby(group_cols, as_index=False)
    .agg(n=("is_sale","sum"), dt=("dt","sum"))
)

# map state columns to a clean design matrix of indicators (0/1 for each intervention)
X_cols = []
for itype in intervention_types:
    col = f"on_{itype}"
    data[col] = data[f"state_{itype}"].astype(int)
    X_cols.append(col)

data.head(10)

,client_id,state_email_nudge,state_onsite_school_visit,state_teacher_webinar,state_virtual_workshop,state_welcome_kit,n,dt,on_email_nudge,on_onsite_school_visit,on_teacher_webinar,on_virtual_workshop,on_welcome_kit
0,0,0,0,0,0,0,13,214.0,0,0,0,0,0
1,0,0,0,0,0,1,2,43.0,0,0,0,0,1
2,0,0,0,0,1,1,6,61.0,0,0,0,1,1
3,0,0,1,0,1,1,7,48.0,0,1,0,1,1
4,1,0,0,0,0,0,7,168.0,0,0,0,0,0
5,1,0,0,1,0,0,18,198.0,0,0,1,0,0
6,2,0,0,0,0,0,9,227.0,0,0,0,0,0
7,2,1,0,0,0,0,4,139.0,1,0,0,0,0
8,3,0,0,0,0,0,15,354.0,0,0,0,0,0
9,3,0,0,0,0,1,2,12.0,0,0,0,0,1


In [ ]:
# Minimal arrays for the model
X = data[X_cols].to_numpy(dtype=float)                 # (N, K) intervention indicators
dt = data['dt'].to_numpy(dtype=float)                  # (N,)
n  = data['n'].to_numpy(dtype=int)                     # (N,)

# map client_id → 0..S-1
client_ids = pd.Index(data['client_id'].unique())
client_index = {cid:i for i,cid in enumerate(client_ids)}
client_idx = data['client_id'].map(client_index).to_numpy(dtype=int)

S = len(client_ids)    # schools
K = len(X_cols)        # interventions

with pm.Model() as model:
    # per-school baseline daily rate, constrained to a reasonable band (as in their nb)
    lam0 = pm.Uniform('lam0', lower=1/60, upper=1/20, shape=S)   # ≈ 0.017..0.05 per day

    # non-negative intervention lifts per day
    alpha = pm.Uniform('alpha', lower=0.0, upper=0.2, shape=K)

    # rate per day for each aggregated row
    lam = pm.Deterministic('lam', lam0[client_idx] + pm.math.dot(X, alpha))

    # expected counts over exposure dt
    mu = pm.Deterministic('mu', lam * dt + 1e-9)  # tiny epsilon avoids log(0)

    # Poisson likelihood
    n_obs = pm.Poisson('n_obs', mu=mu, observed=n)

    idata = pm.sample(draws=1000, tune=1000, chains=2, target_accept=0.9, random_seed=42)

# Posterior summary for baselines and lifts
display(az.summary(idata, var_names=['lam0','alpha'], round_to=4))

# Convert daily lifts to “extra activations per month” (×30)
alpha_draws = idata.posterior['alpha'].stack(sample=('chain','draw')).values  # (chains, draws, K) → stack
alpha_draws = np.moveaxis(alpha_draws, -1, 0).reshape(K, -1)                  # → (K, samples)
monthly = 30.0 * alpha_draws

q1 = pd.DataFrame({
    'intervention': intervention_types,
    'monthly_extra_mean': monthly.mean(axis=1),
    'hdi_95_lower': np.quantile(monthly, 0.025, axis=1),
    'hdi_95_upper': np.quantile(monthly, 0.975, axis=1),
}).sort_values('monthly_extra_mean', ascending=False)
display(q1)

# Onsite ≥ 2× Virtual probability (if both exist)
def _find(cands, names):
    names_l = [s.lower() for s in names]
    for c in cands:
        if c in names_l:
            return names_l.index(c)
    return None

names = [f'on_{it}' for it in intervention_types]
i_on  = _find(['onsite','on_site','visit','school_visit'], names)
i_v   = _find(['virtual','workshop','remote','online'],    names)

if i_on is not None and i_v is not None:
    p = float((monthly[i_on] >= 2.0 * monthly[i_v]).mean())
    print(f"P(onsite ≥ 2× virtual) = {p:.3f}")


Output()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
lam0[0],0.0430,0.0053,0.0333,0.0500,0.0001,0.0001,2196.0535,998.4159,1.0019
lam0[1],0.0386,0.0072,0.0264,0.0500,0.0001,0.0002,2137.9028,897.3864,1.0059
lam0[2],0.0354,0.0076,0.0231,0.0499,0.0002,0.0002,2070.1246,756.7918,1.0042
lam0[3],0.0403,0.0064,0.0289,0.0500,0.0001,0.0001,2016.3665,1057.2194,1.0031
lam0[4],0.0305,0.0082,0.0167,0.0445,0.0002,0.0002,1602.0804,948.1569,1.0036
...,...,...,...,...,...,...,...,...,...
alpha[0],0.0006,0.0005,0.0000,0.0015,0.0000,0.0000,1834.0960,1023.9485,1.0001
alpha[1],0.0317,0.0020,0.0281,0.0355,0.0000,0.0000,1619.0787,1410.0562,1.0007
alpha[2],0.0440,0.0017,0.0409,0.0472,0.0000,0.0000,2088.5186,1501.1320,1.0007
alpha[3],0.0245,0.0018,0.0208,0.0276,0.0000,0.0000,2283.7256,1607.5110,1.0006


,intervention,monthly_extra_mean,hdi_95_lower,hdi_95_upper
4,welcome_kit,0.722351,0.002525,1.370309
0,email_nudge,0.721998,0.002487,1.380159
2,teacher_webinar,0.721702,0.002594,1.376641
3,virtual_workshop,0.721692,0.001925,1.376342
1,onsite_school_visit,0.721659,0.003214,1.378207


In [ ]:
# Q1: posterior summary for "extra activations per month" by intervention
# Uses idata.posterior['alpha'] (per-day lifts ≥ 0); multiplies by 30 to report per-month.


# pull alpha draws: shape (chains, draws, K) → reshape to (K, samples)
alpha = idata.posterior["alpha"].values   # xarray → numpy
# ensure shape is (C, D, K)
if alpha.ndim == 2:  # rare edge cases
    alpha = alpha[None, ...]
C, D, K = alpha.shape
alpha = np.moveaxis(alpha, -1, 0).reshape(K, C*D)   # (K, samples)

monthly = 30.0 * alpha  # convert per-day → per-month

# posterior mean & 95% central interval (2.5–97.5%)
mean  = monthly.mean(axis=1)
lo95  = np.quantile(monthly, 0.025, axis=1)
hi95  = np.quantile(monthly, 0.975, axis=1)

q1 = (pd.DataFrame({
        "intervention": intervention_types,
        "monthly_extra_mean": mean,
        "hdi_95_lower": lo95,
        "hdi_95_upper": hi95,
        "n_draws": monthly.shape[1],
     })
     .sort_values("monthly_extra_mean", ascending=False)
     .reset_index(drop=True))

display(q1.round(4))


,intervention,monthly_extra_mean,hdi_95_lower,hdi_95_upper,n_draws
0,teacher_webinar,1.3194,1.2241,1.4223,2000
1,onsite_school_visit,0.9524,0.8347,1.0692,2000
2,virtual_workshop,0.7345,0.6254,0.8423,2000
3,welcome_kit,0.5857,0.4786,0.6956,2000
4,email_nudge,0.0174,0.0006,0.0593,2000


In [54]:
# Q2 — P(onsite ≥ 2× virtual)  [ROBUST NAME MATCH]

names = [f"on_{it}" for it in intervention_types]
names_lower = [s.lower() for s in names]

def _find_by_substr(substrings, name_list_lower):
    for sub in substrings:
        s = sub.lower()
        for i, nm in enumerate(name_list_lower):
            if s in nm:
                return i
    return None

i_on  = _find_by_substr(["onsite","on_site","school_visit","visit"], names_lower)
i_v   = _find_by_substr(["virtual","workshop","remote","online"],    names_lower)

if i_on is None or i_v is None:
    q2 = pd.DataFrame([{
        "note": "Could not auto-detect both onsite and virtual terms.",
        "detected_terms": ", ".join(names)
    }])
else:
    onsite_draws  = monthly[i_on]     # (samples,)
    virtual_draws = monthly[i_v]
    prob = float((onsite_draws >= 2.0 * virtual_draws).mean())

    q2 = pd.DataFrame([{
        "onsite_term": names[i_on].replace("on_",""),
        "virtual_term": names[i_v].replace("on_",""),
        "prob_onsite_ge_2x_virtual": prob,
        "monthly_onsite_mean": float(onsite_draws.mean()),
        "monthly_virtual_mean": float(virtual_draws.mean()),
        "n_draws": int(monthly.shape[1]),
    }])

display(q2)


,onsite_term,virtual_term,prob_onsite_ge_2x_virtual,monthly_onsite_mean,monthly_virtual_mean,n_draws
0,onsite_school_visit,virtual_workshop,0.0,0.952404,0.734467,2000


In [56]:
# Save CSVs and run a few quick checks

# save
q1_out = "intervention_effects_posterior_summary.csv"
q2_out = "onsite_vs_virtual_probability.csv"
q1.to_csv(q1_out, index=False)
q2.to_csv(q2_out, index=False)
print(f"Saved: {q1_out}\nSaved: {q2_out}")

# sanity checks (lightweight but useful)

# non-negativity of lifts (by construction with Uniform[0, .] priors, but we check anyway)
alpha = alpha_draws  # alias for readability
neg_any = (alpha < 0).any()
print("Check: any negative alpha draws? →", bool(neg_any))

# exposure and counts basic validity
assert (data["dt"] >= 0).all(), "Found negative exposure (dt); check Step 2 sorting/shift."
assert (data["n"]  >= 0).all(), "Found negative counts (n); something is off."

# do we actually have some exposure where each intervention was ON vs OFF? (not required for all schools)
for it in intervention_types:
    on_mask  = data[f"on_{it}"] == 1
    off_mask = data[f"on_{it}"] == 0
    print(f"[{it}] dt when ON = {float(data.loc[on_mask,'dt'].sum()):.1f} days | "
          f"dt when OFF = {float(data.loc[off_mask,'dt'].sum()):.1f} days")

print("\nInterpretation:")
print("- q1: 'monthly_extra_mean' is the best guess for extra activations/month; use the 95% interval for uncertainty.")
print("- q2: 'prob_onsite_ge_2x_virtual' is the decision metric; ≥0.8–0.9 = strong case for onsite; ≤0.5 = unlikely 2×.")

Saved: intervention_effects_posterior_summary.csv
Saved: onsite_vs_virtual_probability.csv
Check: any negative alpha draws? → False
[email_nudge] dt when ON = 26057.0 days | dt when OFF = 156943.0 days
[onsite_school_visit] dt when ON = 27019.0 days | dt when OFF = 155981.0 days
[teacher_webinar] dt when ON = 43392.0 days | dt when OFF = 139608.0 days
[virtual_workshop] dt when ON = 30312.0 days | dt when OFF = 152688.0 days
[welcome_kit] dt when ON = 27679.0 days | dt when OFF = 155321.0 days

Interpretation:
- q1: 'monthly_extra_mean' is the best guess for extra activations/month; use the 95% interval for uncertainty.
- q2: 'prob_onsite_ge_2x_virtual' is the decision metric; ≥0.8–0.9 = strong case for onsite; ≤0.5 = unlikely 2×.


## Results & Recommendations
### Executive summary

#### Question 1 (effect sizes): Extra activations per month (with 95% intervals)

Interpretation: numbers are posterior means; brackets are 95% credible intervals.

- teacher_webinar: ~1.32 extra activations per month,  [1.22,1.42] - largest and tight, clearly positive

- onsite_school_visit: ~0.95, [0.83,1.07] — strong, but below webinar

- virtual_workshop: ~0.73, [0.63,0.84] - solid, cost-effective candidate

- welcome_kit: ~0.59, [0.48,0.70] - — moderate impact

- email_nudge: ~0.02, [0.00,0.06] (negligible)

- Ranking: teacher_webinar > onsite_school_visit > virtual_workshop > welcome_kit » email_nudge.

#### Question 2 (decision metric): Is Onsite at least 2× Virtual?
- Probability that Onsite ≥ 2× Virtual on the “extra activations per month” scale: 0.00. -- 𝑃(Onsite≥2×Virtual)=0.00
- Onsite does not clear the 2× bar in this dataset.
- Why: Onsite mean ≈ 0.95 vs. Virtual mean ≈ 0.73. Ratio ≈ 1.30×, well below 2×.
- Onsite is impactful, but not twice Virtual; the 2× cost-justification threshold is not met.


### Recommendations

- Double down on teacher_webinar. Highest incremental lift with tight uncertainty; make it your default high-impact lever.

- Scale virtual_workshop. Strong, cheaper than onsite, and broadly deployable.

- Use onsite_school_visit selectively. Good effect but not 2× Virtual; reserve for strategic schools (e.g., large cohorts, low adoption without hands-on support).

- De-prioritize email_nudge. Effect is near zero; only consider if the marginal cost is essentially zero or as part of multi-touch sequences.

- Test welcome_kit targeting. Mid-tier lift; experiment with where it complements webinar/virtual.


### Sanity checks - verified

- Exposure (dt) and counts (n) are non-negative; intervals are constructed correctly.

- Interventions have exposure time both ON and OFF across the portfolio (so the model can learn lifts).

- Non-negativity is respected by construction (priors), and posteriors are consistent with Q1.